# Эксперимент 2.5: Микропрофилирование вычислительных затрат

In [ ]:
import time
import urllib.request
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.datasets import load_svmlight_file

from oracles import ExponentialLossL2Oracle
from optimization import nonlinear_conjugate_gradients, hessian_free_newton, lbfgs

In [ ]:
LIBSVM_BASE = "https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets"
data_dir = Path("data/libsvm")
data_dir.mkdir(parents=True, exist_ok=True)
clf_path = data_dir / "a9a"
if not clf_path.exists():
    req = urllib.request.Request(f"{LIBSVM_BASE}/binary/a9a", headers={"User-Agent": "MetOpt-lab/2.0"})
    with urllib.request.urlopen(req, timeout=300) as r, open(clf_path, "wb") as f:
        f.write(r.read())

A, b = load_svmlight_file(clf_path)
A = A.tocsr().astype(np.float64)
b = np.asarray(b, dtype=np.float64).ravel()
if set(np.unique(b)).issubset({0.0, 1.0}):
    b = 2.0 * b - 1.0
m, n = A.shape

def matvec_Ax(x):
    return A.dot(np.asarray(x, dtype=np.float64).ravel())
def matvec_ATx(y):
    return A.T.dot(np.asarray(y, dtype=np.float64).ravel())
def matmat_ATsA(s):
    s = np.asarray(s, dtype=np.float64).ravel()
    return A.T.dot(sparse.diags(s).dot(A)).toarray()

oracle = ExponentialLossL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, b, regcoef=1.0 / m)
x0 = np.zeros(n)
ls = {'method': 'Wolfe', 'c1': 1e-4, 'c2': 0.9, 'alpha_0': 1.0}

In [ ]:
# Приближенное профилирование: делим время на части по вызовам.
# Здесь практичная схема через обертки на оракул.
class TimedOracle:
    def __init__(self, base):
        self.base = base
        self.reset()
    def reset(self):
        self.t_func = 0.0
        self.t_grad = 0.0
        self.t_hv = 0.0
    def func(self, x):
        t0 = time.perf_counter(); out = self.base.func(x); self.t_func += time.perf_counter() - t0; return out
    def grad(self, x):
        t0 = time.perf_counter(); out = self.base.grad(x); self.t_grad += time.perf_counter() - t0; return out
    def hess_vec(self, x, v):
        t0 = time.perf_counter(); out = self.base.hess_vec(x, v); self.t_hv += time.perf_counter() - t0; return out
    def func_directional(self, x, d, alpha):
        return self.func(x + alpha * d)
    def grad_directional(self, x, d, alpha):
        return self.grad(x + alpha * d).dot(d)

def profile_method(run_fn, name):
    tor = TimedOracle(oracle)
    t0 = time.perf_counter()
    run_fn(tor)
    total = time.perf_counter() - t0
    oracle_time = tor.t_func + tor.t_grad + tor.t_hv
    algebra_plus_ls = max(total - oracle_time, 0.0)
    # Грубое деление: считаем, что func/grad_directional в line-search уже учтены в oracle_time.
    return {'name': name, 'oracle': oracle_time, 'algo+ls': algebra_plus_ls, 'total': total}

profiles = []
profiles.append(profile_method(lambda o: nonlinear_conjugate_gradients(o, x0, tolerance=1e-6, max_iter=20, line_search_options=ls), 'NLCG'))
profiles.append(profile_method(lambda o: hessian_free_newton(o, x0, tolerance=1e-6, max_iter=10, line_search_options=ls), 'HFN'))
profiles.append(profile_method(lambda o: lbfgs(o, x0, tolerance=1e-6, max_iter=20, memory_size=10, line_search_options=ls), 'L-BFGS'))
profiles

In [ ]:
names = [p['name'] for p in profiles]
oracle_t = [p['oracle'] for p in profiles]
other_t = [p['algo+ls'] for p in profiles]

x = np.arange(len(names))
plt.figure(figsize=(7, 4))
plt.bar(x, oracle_t, label='Oracle (grad/hess_vec/func)')
plt.bar(x, other_t, bottom=oracle_t, label='Linear algebra + line search infra')
plt.xticks(x, names)
plt.ylabel('time, sec')
plt.title('Microprofiling by method')
plt.legend()
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
Path('fig').mkdir(parents=True, exist_ok=True)
plt.savefig('fig/exp2_5_microprofiling.png', dpi=200, bbox_inches='tight')